# Exercise 2: From the Pair Classifier to FPGA Firmware with _hls4ml_

## Objectives
In this exercise, you will:
1. Rebuild the pair classifier in PyTorch, with its trained weights
2. Choose the fixed-point precision of every layer, and check it in C simulation
3. Estimate the FPGA resources and the latency with Vitis HLS, and see the effect of the reuse factor
4. See how the provided custom kernel is connected to the _hls4ml_ network
5. Get the prebuilt bitstream, and compare the real resources with the estimates

## Instructions
Complete the code cells marked with `# TODO` comments. Follow the hints provided.

## Part 1: Environment Setup
Run the following cells to set up the environment (no changes needed). <br>
This exercise needs the Xilinx tools (Vitis HLS for the synthesis) and XRT (`xclbinutil`). It uses the files written by Exercise 1 in `work/`.

In [ ]:
# Load libraries and import packages (no changes needed)

# General imports
import sys
import re
import shutil
import subprocess
from pathlib import Path

# Numpy and plotting
import numpy as np
import matplotlib.pyplot as plt

# ONNX Runtime, PyTorch and hls4ml
import onnxruntime
import torch
import torch.nn as nn
import hls4ml

# The folder of the course: the one with utils.py and hi_common.py, above the ex/ and sol/ folders
ROOT_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "utils.py").exists() and (folder / "hi_common.py").exists():
        ROOT_DIR = folder
        break
if ROOT_DIR is None:
    raise FileNotFoundError("utils.py and hi_common.py not found: open this notebook from the ex/ or sol/ folder "
                            "of the course")
sys.path.insert(0, str(ROOT_DIR))

# Helpers of the project: configuration, events, the C simulation of the kernel
import hi_common
from pairhead_backends import PairEdgesCSim

# Local file with some useful methods (plots, helpers)
from utils import *

# Configuration and folders
CONFIG = course_config(ROOT_DIR)                # the settings of the course (in utils.py)
WORK_DIR = ROOT_DIR / "work"
DATA_DIR = WORK_DIR / "data"          # written by Exercise 1
HLS_DIR = WORK_DIR / "hls"            # hls4ml projects
KERNEL_DIR = WORK_DIR / "kernel"      # the custom kernel with the hls4ml network
FPGA_DIR = WORK_DIR / "fpga"          # the prebuilt bitstream
FIGURE_DIR = WORK_DIR / "figures"
for folder in (HLS_DIR, KERNEL_DIR, FPGA_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

FPGA_PART = CONFIG["hls"]["part"]     # FPGA part of the Alveo U55C        - DO NOT CHANGE
CLOCK_PERIOD_NS = 1000.0 / 300        # 300 MHz clock                      - DO NOT CHANGE
THRESHOLD = CONFIG["operating_threshold"]
model_config = course_model_config(ROOT_DIR)    # the settings the GTK code reads (in utils.py)
THRESHOLDS = hi_common.thresholds(CONFIG, model_config)   # the thresholds at which decisions are checked
RUN_KERNEL_CSYNTH = True              # C-synthesis of the custom kernel in Part 6 (about 2 minutes)

use_course_style()
tools_found = {tool: shutil.which(tool) is not None for tool in ("vitis_hls", "g++", "xclbinutil")}
print(f"hls4ml {hls4ml.__version__}, torch {torch.__version__}")
print(f"Tools found: {tools_found}")
print("Environment setup correctly!" if all(tools_found.values()) else "Some tools are missing: vitis_hls, g++ and xclbinutil must be in the PATH")

In Exercise 1 you split the trained model, and saved the weights of the pair classifier and some real pair features in `work/data/`. Now the pair classifier goes to the FPGA.

## Part 2: The Pair Classifier in PyTorch
_hls4ml_ reads the model through its PyTorch frontend. The pair classifier is the small MLP of Exercise 0: 64 inputs, 64 hidden neurons with a ReLU, 2 outputs.

In [ ]:
# Define the pair classifier and load the reference pairs (no changes needed)
class PairClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):          # hls4ml names the input layer after this argument: "x"
        return self.fc2(self.relu(self.fc1(x)))


calibration_pairs = np.load(DATA_DIR / "pairs_validation.npz")   # to measure the ranges of the values
test_pairs = np.load(DATA_DIR / "pairs_test.npz")                # to measure the accuracy
calibration_features = calibration_pairs["X"]
test_features = np.ascontiguousarray(test_pairs["X"])
float_logits = test_pairs["logits"]                              # pair_head.onnx on the test pairs
print(f"Calibration pairs: {calibration_features.shape}, test pairs: {test_features.shape}")

### Task 2.1: Load the Trained Weights

In [ ]:
# TODO: load the trained weights of the pair classifier into the PyTorch model

# Hint: np.load(DATA_DIR / 'pair_head_weights.npz') gives the arrays W1, b1, W2, b2 saved by Exercise 1
# Hint: ONNX stores a weight matrix as [inputs, outputs], PyTorch's nn.Linear as [outputs, inputs]:
#       transpose it with .T (and .copy() it, to make it contiguous)
# Hint: inside torch.no_grad(), model.fc1.weight.copy_(torch.from_numpy(array)) copies an array into a parameter

weights = None  # Replace with correct code
model = PairClassifier().eval()
with torch.no_grad():
    # YOUR CODE HERE
    pass

with torch.no_grad():
    torch_logits = model(torch.from_numpy(test_features)).numpy()
print(model)
print(f"PyTorch vs pair_head.onnx on {len(test_features):,} test pairs: largest difference in the logits "
      f"{np.abs(torch_logits - float_logits).max():.1e}")

## Part 3: Fixed-Point Precision
The FPGA does not compute with floats but with **fixed-point** numbers, `ap_fixed<W, I>`: *W* bits in total, *I* of them for the integer part (sign included). It holds values from −2<sup>I−1</sup> to 2<sup>I−1</sup>, in steps of 2<sup>−(W−I)</sup>. For every tensor we choose:
- the **integer bits**, from the largest value the tensor takes on real data, so that nothing overflows
- the **total bits**, a trade-off: more bits are more accurate, fewer bits use fewer FPGA resources

### Task 3.1: The Ranges of the Values

In [ ]:
# TODO: measure the largest absolute value of every tensor of the pair classifier

# Hint: run the model on the calibration pairs inside torch.no_grad(): x = torch.from_numpy(calibration_features);
#       model.fc1(x) gives the hidden values (before the ReLU), model(x) the logits; .numpy() converts them
# Hint: np.abs(array).max() is the largest absolute value of an array; the weights are in `weights`

largest = {"input": None,          # Replace with correct code
           "hidden": None,         # Replace with correct code
           "logits": None,         # Replace with correct code
           "fc1 weight": None,     # Replace with correct code
           "fc1 bias": None,       # Replace with correct code
           "fc2 weight": None,     # Replace with correct code
           "fc2 bias": None}       # Replace with correct code

for name, value in largest.items():
    print(f"{name:11s} largest |value| {value:7.3f}")

### Task 3.2: The Integer Bits
An `ap_fixed<W, I>` holds values up to 2<sup>I−1</sup>: to hold values up to *v* we need the smallest *I* with 2<sup>I−1</sup> > *v*, that is *I* = ⌊log₂ *v*⌋ + 2 (one of the bits is the sign). For the activations we add a 25% margin, in case some event goes a bit further than the calibration ones.

In [ ]:
# TODO: write the function that gives the integer bits (sign included) needed for values up to largest_value

# Hint: I = floor(log2(largest_value)) + 2: use np.floor and np.log2, and return an int
# Hint: return at least 1 (the sign bit), also when largest_value is 0

def integer_bits_for(largest_value):
    # YOUR CODE HERE
    return None  # Replace with correct code

ACTIVATION_MARGIN = 1.25                                          # DO NOT CHANGE
integer_bits = {}
for name, value in largest.items():
    is_activation = name in ("input", "hidden", "logits")
    integer_bits[name] = integer_bits_for(value * ACTIVATION_MARGIN if is_activation else value)
    print(f"{name:11s} largest |value| {value:7.3f}  ->  {integer_bits[name]} integer bits")

The cell below turns integer bits and total bits into _hls4ml_ types. We use two total widths, one for the **data** (input, hidden values, logits) and one for the **weights** and biases. Every type rounds to the nearest value (`AP_RND_CONV`) and saturates instead of wrapping around (`AP_SAT`).

In [ ]:
# The types of the pair classifier for given weight bits and data bits (no changes needed)
def fixed_type(total_bits, integer_bits):
    return f"ap_fixed<{total_bits},{integer_bits},AP_RND_CONV,AP_SAT>"


def precision(weight_bits, data_bits):
    return {"input": fixed_type(data_bits, integer_bits["input"]),
            "hidden": fixed_type(data_bits, integer_bits["hidden"]),
            "logits": fixed_type(data_bits, integer_bits["logits"]),
            "w1": fixed_type(weight_bits, integer_bits["fc1 weight"]),
            "b1": fixed_type(weight_bits, integer_bits["fc1 bias"]),
            "w2": fixed_type(weight_bits, integer_bits["fc2 weight"]),
            "b2": fixed_type(weight_bits, integer_bits["fc2 bias"])}


print_dict(precision(16, 18))

## Part 4: Convert, Compile and Simulate
### Task 4.1: The _hls4ml_ Model
`config_from_pytorch_model` creates a configuration with one entry per layer, `hls_config["LayerName"]`: `x` (the input), `fc1`, `relu` and `fc2`. Each entry has a `Precision` dictionary: complete the function below so that it sets the types of every layer.

In [ ]:
# TODO: set the precision of every layer of the hls4ml configuration

# Hint: layers['x']['Precision']['result'] is the type of the input; layers['relu']['Precision']['result'] the one
#       of the hidden values after the ReLU
# Hint: fc1 and fc2 have 'weight', 'bias', 'result' and 'accum' (the sums): set 'accum' to 'auto', so that hls4ml
#       makes the sums wide enough to be exact
# Hint: the types are in the dictionary `types`: types['input'], types['hidden'], types['logits'], types['w1'], ...

def make_hls_model(types, output_dir, strategy="Latency", reuse_factor=1):
    hls_config = hls4ml.utils.config_from_pytorch_model(model, [(None, 64)], granularity="name", backend="Vivado",
                                                        default_precision="ap_fixed<16,6>",
                                                        default_reuse_factor=reuse_factor)
    hls_config["Model"]["Strategy"] = strategy
    layers = hls_config["LayerName"]

    # YOUR CODE HERE

    for layer_name in ("fc1", "fc2"):
        layers[layer_name]["ReuseFactor"] = reuse_factor
        layers[layer_name]["Strategy"] = strategy
    return hls4ml.converters.convert_from_pytorch_model(
        model, input_shape=[(None, 64)], hls_config=hls_config, output_dir=str(output_dir), backend="Vivado",
        io_type="io_parallel", part=FPGA_PART, clock_period=CLOCK_PERIOD_NS, project_name="myproject")

The cell below defines how we compare the fixed-point model with the float one, on the test pairs. A configuration is **accepted** if, at every threshold, no decision changes for pairs further than 10⁻³ from the threshold, the probability p₁ moves by at most 2 × 10⁻³ (2 × 10⁻⁴ on average), and no input saturates.

In [ ]:
# Compare the fixed-point model with the float one (no changes needed)
MAX_DP, MEAN_DP, BAND = 2e-3, 2e-4, 1e-3        # acceptance limits - DO NOT CHANGE


def probability_of_edge(logits):
    return softmax(logits)[:, 1]


def compare_with_float(fixed_logits, types):
    p_fixed = probability_of_edge(fixed_logits)
    p_float = probability_of_edge(float_logits)
    differences = np.abs(p_fixed - p_float)
    changes, changes_far_from_threshold = 0, 0
    for threshold in THRESHOLDS:
        changed = (p_fixed > threshold) != (p_float > threshold)
        changes += int(changed.sum())
        changes_far_from_threshold += int((changed & (np.abs(p_float - threshold) > BAND)).sum())
    total_bits, integer = [int(number) for number in re.findall(r"\d+", types["input"])[:2]]
    largest_input = 2.0 ** (integer - 1) - 2.0 ** (integer - total_bits)
    result = {"max_abs_dp": float(differences.max()), "mean_abs_dp": float(differences.mean()),
              "decision_changes": changes, "changes_beyond_band": changes_far_from_threshold,
              "saturated_inputs": int((np.abs(test_features) > largest_input).sum())}
    result["accepted"] = (result["changes_beyond_band"] == 0 and result["max_abs_dp"] <= MAX_DP
                          and result["mean_abs_dp"] <= MEAN_DP and result["saturated_inputs"] == 0)
    return result

### Task 4.2: A First Conversion
Convert the pair classifier with a generous precision, 16 bits for the weights and 18 for the data, and run its **C simulation**: `compile()` builds a C++ emulation of the firmware, and `predict()` runs it, with exactly the fixed-point arithmetic of the FPGA.

In [ ]:
# TODO: convert the pair classifier with precision(16, 18), compile it, and run its C simulation on the test pairs

# Hint: make_hls_model(types, output_dir) with output_dir=HLS_DIR / 'first_try'
# Hint: hls_model.compile() builds the C simulation, hls_model.predict(test_features) runs it
# Hint: predict returns a flat array: reshape it to [K, 2] with .reshape(-1, 2)

first_types = precision(16, 18)
with output_to_log_file(HLS_DIR / "first_try.log"):     # hls4ml's messages go to this file
    first_model = None  # Replace with correct code
    # YOUR CODE HERE: compile it

first_logits = None     # Replace with correct code: the C simulation of the test pairs, [K, 2]

print(f"C simulation of {len(first_logits):,} test pairs vs float:")
print_dict(compare_with_float(first_logits, first_types))

### Task 4.3: The Precision Scan
How few bits can we use? The cell below converts, compiles and simulates six configurations (about two minutes); then choose the smallest accepted one.

In [ ]:
# Precision scan: convert, compile and simulate six configurations (no changes needed)
CONFIGURATIONS = [(8, 18), (10, 18), (12, 12), (12, 14), (12, 16), (14, 18)]   # (weight bits, data bits)
scan_results = {}
for weight_bits, data_bits in CONFIGURATIONS:
    name = f"w{weight_bits}_d{data_bits}"
    with output_to_log_file(HLS_DIR / f"scan_{name}.log"):
        scan_model = make_hls_model(precision(weight_bits, data_bits), HLS_DIR / f"scan_{name}")
        scan_model.compile()
    scan_logits = np.asarray(scan_model.predict(test_features), np.float32).reshape(-1, 2)
    scan_results[(weight_bits, data_bits)] = compare_with_float(scan_logits, precision(weight_bits, data_bits))
    result = scan_results[(weight_bits, data_bits)]
    print(f"{name:8s} max|Δp| {result['max_abs_dp']:.1e}  mean {result['mean_abs_dp']:.1e}  "
          f"decision changes {result['decision_changes']:5d} (beyond the band: {result['changes_beyond_band']:4d})  "
          f"saturated {result['saturated_inputs']}  {'ACCEPTED' if result['accepted'] else '-'}")

In [ ]:
# TODO: choose the smallest accepted configuration

# Hint: keep the (weight bits, data bits) keys of scan_results whose result is accepted
# Hint: the smallest one has the fewest total bits (weight + data); between equal totals, take the fewest data bits:
#       min(accepted, key=...) with a function that returns (weight + data, data)

accepted = None                  # Replace with correct code: the accepted (weight bits, data bits)
WEIGHT_BITS, DATA_BITS = None    # Replace with correct code

TYPES = precision(WEIGHT_BITS, DATA_BITS)
print(f"Selected: w{WEIGHT_BITS}_d{DATA_BITS}")
plot_precision_scan(scan_results, (WEIGHT_BITS, DATA_BITS), MAX_DP, save_to=FIGURE_DIR / "precision_scan.png")
print("Plot saved as 'work/figures/precision_scan.png'")

### Task 4.4: The Final Model
The cell below converts the pair classifier at the selected precision and looks at what _hls4ml_ produced: the types of every layer, and the C++ of the firmware (`firmware/myproject.cpp` calls one function per layer).

In [ ]:
# Convert the final model and inspect it (no changes needed)
FINAL_PROJECT = HLS_DIR / "pair_classifier"
shutil.rmtree(FINAL_PROJECT, ignore_errors=True)
with output_to_log_file(HLS_DIR / "pair_classifier.log"):
    hls_model = make_hls_model(TYPES, FINAL_PROJECT)
    hls_model.compile()
show_hls_layers(hls_model)

firmware_dir = FINAL_PROJECT / "firmware"
print(f"\nfirmware/: {sorted(path.name for path in firmware_dir.iterdir())}")
print((firmware_dir / "myproject.cpp").read_text().split("// hls-fpga-machine-learning insert layers")[1].strip()[:700])

hls_logits = np.asarray(hls_model.predict(test_features), np.float32).reshape(-1, 2)
print(f"\nC simulation vs float:")
print_dict(compare_with_float(hls_logits, TYPES))

In [ ]:
# Visualize the fixed-point error and the types (no changes needed)
plot_fixed_point_error(probability_of_edge(hls_logits), probability_of_edge(float_logits),
                       f"w{WEIGHT_BITS}_d{DATA_BITS}", save_to=FIGURE_DIR / "fixed_point_error.png")
plot_bit_layout(TYPES, save_to=FIGURE_DIR / "bit_layout.png")
print("Plots saved as 'work/figures/fixed_point_error.png' and 'work/figures/bit_layout.png'")

## Part 5: Resources and Latency
The **C-synthesis** of Vitis HLS turns the C++ into a hardware design, and reports, for the U55C at 300 MHz:
- the **latency**: the clocks from an input to its output
- the **interval**: the clocks between two new inputs (1: a new pair every clock)
- the **resources**: DSP slices (the multipliers), LUTs, flip-flops (FF), block RAMs

Two settings of _hls4ml_ decide how parallel the design is:
- `Strategy = "Latency"`: every multiplication gets its own multiplier
- `Strategy = "Resource"` with a **reuse factor** R: each multiplier is shared by R multiplications, so R times fewer multipliers, but a new input only every R clocks

**Important:** each synthesis takes a few minutes. Run the two tasks now!

### Task 5.1: Synthesize the Latency Version

In [ ]:
# TODO: run the C-synthesis of the pair classifier with the Latency strategy and reuse factor 1

# Hint: make_hls_model(TYPES, output_dir, strategy='Latency', reuse_factor=1), with output_dir=SYNTH_DIR
# Hint: hls_model.write() writes the project, hls_model.build(csim=False, synth=True, export=False) runs Vitis HLS

SYNTH_DIR = HLS_DIR / "csynth_latency_rf1"
shutil.rmtree(SYNTH_DIR, ignore_errors=True)                # start from an empty folder
with output_to_log_file(HLS_DIR / "csynth_latency_rf1.log"):    # the long Vitis HLS log goes to this file
    # YOUR CODE HERE
    pass

synthesis_results = {}
synthesis_results["Latency, RF 1"] = read_csynth_report(SYNTH_DIR / "myproject_prj" / "solution1" / "syn" / "report"
                                                        / "myproject_csynth.xml")
print_dict(synthesis_results["Latency, RF 1"])

### Task 5.2: Synthesize the Resource Version

In [ ]:
# TODO: run the C-synthesis of the pair classifier with the Resource strategy and reuse factor 16

# Hint: the same as Task 5.1, with strategy='Resource', reuse_factor=16 and output_dir=SYNTH_DIR

SYNTH_DIR = HLS_DIR / "csynth_resource_rf16"
shutil.rmtree(SYNTH_DIR, ignore_errors=True)
with output_to_log_file(HLS_DIR / "csynth_resource_rf16.log"):
    # YOUR CODE HERE
    pass

synthesis_results["Resource, RF 16"] = read_csynth_report(SYNTH_DIR / "myproject_prj" / "solution1" / "syn"
                                                          / "report" / "myproject_csynth.xml")
print_dict(synthesis_results["Resource, RF 16"])

In [ ]:
# Compare the two versions (no changes needed)
plot_synthesis_variants(synthesis_results, save_to=FIGURE_DIR / "synthesis_variants.png")
print("Plot saved as 'work/figures/synthesis_variants.png'")

The Latency version takes a new pair every clock, but every one of the 4,224 multiplications of a pair gets its own hardware: 3,117 DSP slices, and the others are built from LUTs (the weights are constants, and a multiplication by a constant can often be done with a few shifts and additions). The Resource version with reuse factor 16 uses about 12 times fewer DSP slices, but takes a new pair only every 16 clocks. This is the main trade-off of FPGA design: **resources against speed**.

## Part 6: Connecting the Custom Kernel
At one pair per clock, the pair classifier could score a median event in about 4 µs, but only if the pairs reach it that fast. If the pairs are built on the CPU, every pair is 64 floats: for a median event, about 340 kB to send to the card. The data the pairs are made from is much smaller: the 51 embeddings, 6.5 kB.

In [ ]:
# Visualize the data sent to the card per event (no changes needed)
plot_data_per_event(save_to=FIGURE_DIR / "data_per_event.png")
print("Plot saved as 'work/figures/data_per_event.png'")

So the FPGA receives the embeddings and builds the pairs itself, in a **custom C++ kernel** (`custom_kernel/pairedges_fact_kernel.cpp`). **This kernel is already written for you**: you do not need to change it, but it is worth seeing how it is connected to the _hls4ml_ network.

The kernel does not call _hls4ml_'s layer functions: it has its own loops, so that it can compute the first layer once per hit instead of once per pair (z = U[i] + V[j]). Inside the loops it uses what _hls4ml_ generated: the fixed-point types, the quantized weights, the multiplications and the roundings. So its results are **bit-identical** to the _hls4ml_ model you just built.

In [ ]:
# Visualize the stages of the kernel (no changes needed)
plot_kernel_pipeline(save_to=FIGURE_DIR / "kernel_pipeline.png")
print("Plot saved as 'work/figures/kernel_pipeline.png'")

To use the _hls4ml_ code, the kernel needs the names of its objects (configurations, types, weight arrays), which _hls4ml_ numbers by layer position. The cell below reads them from the generated `myproject.cpp`, writes them in a small header, copies the _hls4ml_ firmware and the kernel source in one folder, and compiles the C simulation of the kernel.

In [ ]:
# Connect the kernel to the hls4ml firmware, and compile its C simulation (no changes needed)
top_function_lines = (firmware_dir / "myproject.cpp").read_text().splitlines()


def layer_call(layer_name):
    # in myproject.cpp each layer is one call, e.g. nnet::dense<input_t, layer2_t, config2>(x, layer2_out, w2, b2); // fc1
    for line in top_function_lines:
        if line.strip().endswith(f"// {layer_name}"):
            template_arguments = line.split("<", 1)[1].split(">(", 1)[0].split(", ")
            call_arguments = line.split(">(", 1)[1].split(");", 1)[0].split(", ")
            return template_arguments, call_arguments
    raise ValueError(f"layer {layer_name} not found in myproject.cpp")


fc1_template, fc1_arguments = layer_call("fc1")
relu_template, _ = layer_call("relu")
fc2_template, fc2_arguments = layer_call("fc2")
kernel_names = {"PH_FC1_CONFIG": fc1_template[2], "PH_FC2_CONFIG": fc2_template[2],   # configurations of the layers
                "PH_FC1_OUT_T": fc1_template[1], "PH_RELU_OUT_T": relu_template[1],     # types
                "PH_W1": fc1_arguments[2], "PH_B1": fc1_arguments[3],                  # weight and bias arrays
                "PH_W2": fc2_arguments[2], "PH_B2": fc2_arguments[3]}
for array in ("PH_W1", "PH_B1", "PH_W2", "PH_B2"):
    kernel_names[f"{array}_FILE"] = f'"{kernel_names[array]}.txt"'
header = "// hls4ml names used by the kernel\n#pragma once\n"
header += "".join(f"#define {macro} {value}\n" for macro, value in kernel_names.items())
print(header)

KERNEL_SOURCE = ROOT_DIR / "custom_kernel" / "pairedges_fact_kernel.cpp"
shutil.rmtree(KERNEL_DIR, ignore_errors=True)
shutil.copytree(firmware_dir, KERNEL_DIR / "firmware", ignore=shutil.ignore_patterns("*.so", "*_axi.*"))
(KERNEL_DIR / "firmware" / "pairhead_names.h").write_text(header)
shutil.copy(KERNEL_SOURCE, KERNEL_DIR / KERNEL_SOURCE.name)

compile_command = (f"g++ -O2 -fPIC -shared -std=c++14 -I firmware -I firmware/ap_types -DPE_N_MAX=256 "
                   f"-DWEIGHTS_DIR='\"{KERNEL_DIR / 'firmware' / 'weights'}\"' {KERNEL_SOURCE.name} -o libpairedges_csim.so")
build = subprocess.run(["bash", "-c", compile_command], cwd=KERNEL_DIR, capture_output=True, text=True)
print("Kernel C simulation:", "compiled" if build.returncode == 0 else build.stderr[-1500:])

The cell below checks the connection on 100 test events: the kernel receives the embeddings and returns the **candidates** (the pairs that can pass the threshold, with their integer logits); the reference builds the pairs on the CPU and runs your _hls4ml_ model on them. They must give the same pairs with the same logits, bit for bit.

In [ ]:
# Check the kernel against the hls4ml model (no changes needed)
encoder = onnxruntime.InferenceSession(str(WORK_DIR / "onnx" / "encoder.onnx"), providers=["CPUExecutionProvider"])
kernel_simulation = PairEdgesCSim(KERNEL_DIR)

logit_bits, logit_integer_bits = [int(number) for number in re.findall(r"\d+", TYPES["logits"])[:2]]
FRACTION_BITS = logit_bits - logit_integer_bits             # integer logit = logit x 2^FRACTION_BITS
CANDIDATE_THRESHOLD = hi_common.candidate_thr_raw(THRESHOLD, FRACTION_BITS)

same_candidates, total_pairs, total_candidates = True, 0, 0
for event in hi_common.load_events(CONFIG, model_config, "test", 100):
    embeddings = encoder.run(None, {"data": event["features"]})[0][0]
    n = len(embeddings)
    rows, columns = np.triu_indices(n)

    # reference: pairs built on the CPU, C simulation of the hls4ml model, the kernel's test on the integer logits
    pair_features = np.ascontiguousarray(np.concatenate([embeddings[rows], embeddings[columns]], axis=1))
    reference_logits = np.asarray(hls_model.predict(pair_features), np.float32).reshape(-1, 2)
    integer_logits = np.round(reference_logits * 2 ** FRACTION_BITS).astype(np.int64)
    reference_rows = np.flatnonzero(integer_logits[:, 1] - integer_logits[:, 0] > CANDIDATE_THRESHOLD)

    # the kernel: embeddings in, candidates out
    candidate_words = kernel_simulation.run([embeddings], CANDIDATE_THRESHOLD)[0][0]
    first, second, kernel_logits = hi_common.unpack_candidates(candidate_words, FRACTION_BITS)
    kernel_rows = first * n - first * (first - 1) // 2 + (second - first)       # row of (i, j) in the pair list
    same_candidates &= np.array_equal(kernel_rows, reference_rows)
    same_candidates &= np.array_equal(kernel_logits, reference_logits[reference_rows])
    total_pairs += len(rows)
    total_candidates += len(candidate_words)

print(f"{total_pairs:,} pairs -> {total_candidates:,} candidates ({total_candidates / total_pairs:.0%})")
print(f"Kernel == hls4ml model, bit for bit: {same_candidates}")

The C-synthesis of the whole kernel (about 2 minutes) shows that every loop takes a new item every clock (`Final II = 1`), and gives the resources of the kernel.

In [ ]:
# C-synthesis of the kernel (no changes needed)
kernel_synthesis = None
if RUN_KERNEL_CSYNTH and tools_found["vitis_hls"]:
    (KERNEL_DIR / "csynth.tcl").write_text(f'''open_project -reset pairedges_prj
set_top pairedges
add_files {KERNEL_SOURCE.name} -cflags "-I firmware -DPE_N_MAX=256"
open_solution -reset solution1 -flow_target vitis
set_part {FPGA_PART}
create_clock -period {CLOCK_PERIOD_NS:.3f} -name default
config_interface -m_axi_alignment_byte_size 64 -m_axi_max_widen_bitwidth 512
csynth_design
exit
''')
    with open(KERNEL_DIR / "vitis_hls.log", "w") as log_file:
        subprocess.run(["vitis_hls", "-f", "csynth.tcl"], cwd=KERNEL_DIR, stdout=log_file, stderr=subprocess.STDOUT)
    for line in (KERNEL_DIR / "vitis_hls.log").read_text().splitlines():
        if "Pipelining result" in line:
            print(line.replace("INFO: [HLS 200-1470] ", ""))
    kernel_synthesis = read_csynth_report(KERNEL_DIR / "pairedges_prj" / "solution1" / "syn" / "report"
                                          / "pairedges_csynth.xml")
    print({resource: kernel_synthesis[resource] for resource in U55C_RESOURCES})

## Part 7: The Prebuilt Bitstream
The last step, `v++ --link`, places the kernel into the U55C platform (the part of the FPGA that handles PCIe, the card memory and the clocks), then runs synthesis, placement and routing in Vivado, and writes the bitstream, an `.xclbin` file. **It takes about 2 hours**, so we built it for you, from exactly this kernel and this _hls4ml_ model (precision w12_d14).

The cell below downloads the bitstream from the storage of the course (49 MB) and checks that it is the right file. `PREBUILT_BITSTREAM` (in `utils.py`) describes it: how it was linked, the steps of the build, and its settings.

In [ ]:
# Get the prebuilt bitstream (no changes needed)
bitstream_file = download_bitstream(FPGA_DIR)
prebuilt = PREBUILT_BITSTREAM
print(f"\nLink configuration:\n{prebuilt['link_config']}")
print(f"Link command: {prebuilt['link_command']}\n")
print("\n".join(prebuilt["build_steps"]))

### Task 7.1: Inspect the Bitstream
`xclbinutil` (from XRT) describes an `.xclbin`: its kernels, the memory they use, and the clock the kernel really reached after routing.

In [ ]:
# TODO: read the description of the bitstream with xclbinutil

# Hint: subprocess.run(['xclbinutil', '--info', '--input', str(path)], capture_output=True, text=True).stdout
# Hint: the bitstream is bitstream_file

xclbin_info = None  # Replace with correct code

kernel_clock = re.search(r"ulp_ucs_aclk_kernel_00[\s\S]{0,300}?Requested Freq:\s*([0-9.]+)\s*MHz"
                         r"[\s\S]{0,100}?Achieved Freq:\s*([0-9.]+)\s*MHz", xclbin_info)
print(f"Kernel clock: requested {kernel_clock.group(1)} MHz, achieved {kernel_clock.group(2)} MHz")
memories = sorted(set(re.findall(r"Memory:\s+(HBM\[\d+\])", xclbin_info)))
print(f"Compute unit pairedges_1: {'pairedges_1' in xclbin_info}")
print(f"Card memory used by the kernel: {memories}")

### Task 7.2: Estimates vs Reality
How good were the Vitis HLS estimates? `PREBUILT_BITSTREAM` has the resources the kernel really uses after place and route.

In [ ]:
# Compare the Vitis HLS estimate with the real kernel (no changes needed)
estimate = kernel_synthesis if kernel_synthesis is not None else prebuilt["hls_estimate"]
routed = prebuilt["routed"]
plot_estimate_vs_routed(estimate, routed, save_to=FIGURE_DIR / "estimate_vs_routed.png")
print("Plot saved as 'work/figures/estimate_vs_routed.png'")

The DSP count is almost exact (359 estimated, 360 used), because every multiplier is placed as written. LUTs, flip-flops and memories are overestimated: Vitis HLS does not know how Vivado will optimise the logic, and it counts the worst case. The estimates are useful to compare designs and to check that a design fits, but the real numbers only come from place and route.

Finally, the cell below checks that the precision you selected is the one of the bitstream: Exercise 3 runs the bitstream and compares it with the C simulation of the kernel you compiled in Part 6.

In [ ]:
# Check the precision (no changes needed)
same_precision = TYPES == prebuilt["precision"]
print(f"Your precision w{WEIGHT_BITS}_d{DATA_BITS}, bitstream {prebuilt['precision_name']}: "
      f"{'the same' if same_precision else 'DIFFERENT: go back to Task 4.3'}")

In **Exercise 3** you will load the bitstream on the Alveo U55C, run the hybrid model (the encoder on the CPU, the pair building and the pair classifier on the FPGA), check it against the C simulation and the float model, and measure how fast it is compared with the CPU.